# Step 1. Business Understanding – IMDB Dataset

The IMDB dataset contains information about movies, including attributes like title, genre, director, cast, ratings, and reviews. The goal is to develop a movie similarity recommendation system that suggests movies similar to a given movie based on genre, release year, user ratings, and popularity. This system will help users discover movies that match their preferences without relying on complex user behavior tracking or reviews

## Determine Business Objectives / Data Mining Goals

- **Objective**:
  - Recommend movies that are similar to a given movie based on genre, release year, ratings, and popularity.


## Assess Situation

### **Resources:**  
- **IMDB dataset**, including:  
  - **Genre** (e.g., Action, Drama, Comedy).  
  - **Release Year** (to ensure recommendations match the era of the given movie).  
  - **User Ratings** (e.g., IMDb score).  
  - **Popularity** (e.g., number of votes).  
- **Data Source:**  
  - [IMDB Open Database](https://www.imdb.com/interfaces/)

### **Constraints:**  
- Older movies may have fewer ratings, affecting similarity scores.  
- Genre overlaps may result in repetitive recommendations.  
- Popularity bias (highly rated blockbusters might dominate recommendations).  

## Project Plan

### **Deliverables:**  
- **Feature Engineering:** Multi-hot-encoding of genres, normalizing ratings and popularity scores.  
- **Movie Similarity Model:** Using a combination of genre, year, ratings, and popularity.  
- **Interactive Recommendation System:** Users input a movie, and similar movies are returned.  

# Step 2: Data Understanding / Exploratory Data Analysis (EDA)

## Load Libraries

In [ ]:
from tqdm import tqdm
import os.path
import urllib.request
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from IPython.display import display
import plotly.express as px

## Define Parameters

In [ ]:
# URL to IMDB data base for non-commercial use
URL = 'https://datasets.imdbws.com/'
DATA_FOLDER = './data/'
DATA_FILES = ['title.akas.tsv.gz', 'title.basics.tsv.gz', 'title.crew.tsv.gz', 'title.ratings.tsv.gz']
DF_FILE = 'imdb.pkl'

In [ ]:
FORCE_DOWNLOAD = False
FORCE_RECREATE_DF = False

## Collect Initial Data

In [ ]:
if not os.path.exists(DATA_FOLDER):
    os.mkdir(DATA_FOLDER)

In [ ]:
# Iterate over all files in the DATA_FILES list
for file in tqdm(DATA_FILES):
    # Check if FORCE_DOWNLOAD is set to 'True' or if the file wasn't downloaded yet only proceed if one of the conditions is met
    if FORCE_DOWNLOAD or not os.path.isfile(DATA_FOLDER+file):
        # Download file and write to disk
        urllib.request.urlretrieve(url=URL+file, filename=DATA_FOLDER+file)

The first time you execute this cell, it will take a couple of minutes to create the dataframe. We are saving the dataframe afterwards, s.t. on every next execution, we can load it quickly.

In [ ]:
# Check if FORCE_RECREATE is set to 'True' or if the DF_FILE doesn't exists and only proceed if one of the conditions is met
if FORCE_RECREATE_DF or not os.path.isfile(DATA_FOLDER+DF_FILE):
    # define dtypes to use when reading the csv files
    dtypes = {
        # title.akas.tsv.gz
        'titleId': 'string',
        'title': 'string',
        'region': 'category',

        # title.basics.tsv.gz
        'tconst': 'string',
        'titleType': 'category',
        'originalTitle': 'string',
        'isAdult': 'boolean',
        'startYear': 'Int16',
        'genres': 'string',

        # title.crew.tsv.gz
        'directors': 'string',
        'writers': 'string',

        # title.ratings.tsv.gz
        'averageRating': 'float32',
        'numVotes': 'int32'
    }

    # read and merge data from the four files
    df = pd.read_csv(DATA_FOLDER+'title.akas.tsv.gz', sep='\t', usecols=['titleId', 'region', 'title'], dtype=dtypes, na_values=r'\N')
    df = df.loc[(df.region=='DE')].drop(columns=['region'])
    print("title.akas loaded")
    display(df)

    df_basics = pd.read_csv(DATA_FOLDER+'title.basics.tsv.gz', usecols=['tconst', 'titleType', 'originalTitle', 'isAdult', 'startYear', 'genres'], sep='\t', na_values=r'\N')
    df = pd.merge(left=df, right=df_basics, left_on='titleId', right_on='tconst', how='left')
    del df_basics
    print("\ntitle.basics loaded and merged")
    display(df)

    df_crew = pd.read_csv(DATA_FOLDER+'title.crew.tsv.gz', sep='\t', dtype=dtypes, na_values=r'\N')
    df = pd.merge(left=df, right=df_crew, left_on='titleId', right_on='tconst', how='left')
    del df_crew
    print("\ntitle.crew loaded and merged")
    display(df)

    df_ratings = pd.read_csv(DATA_FOLDER+'title.ratings.tsv.gz', sep='\t', dtype=dtypes, na_values=r'\N')
    df = pd.merge(left=df, right=df_ratings, left_on='titleId', right_on='tconst', how='left')
    del df_ratings
    print("\ntitle.ratings loaded and merged")
    display(df)

    columns = ['titleId', 'title', 'originalTitle', 'startYear', 'genres', 'directors', 'writers', 'averageRating', 'numVotes']
    df = df.loc[(df.titleType == 'movie') & (df.isAdult == 0), columns].replace(r'\N', np.nan).dropna()
    df = df.convert_dtypes()

    df.to_pickle(DATA_FOLDER+DF_FILE)
    print("\nDataFrame filtered and saved")
    display(df)

else:
    df = pd.read_pickle(DATA_FOLDER+DF_FILE)

## ❗ **Homework** ❗  
1. Investigate the original DataFrame (`df`). Especially focus on the features `startYear`, `genres`, `averageRating` and `numVotes`. What do you notice?


## Describe Data

In [ ]:
# ...

**Observations:**
- ...

**Conclusions:**
- ...


## Explore Data

In [ ]:
# ...

**Observations:**
- ...

**Conclusions:**
- ...


# Step 3: Data Preparation

## Select Data

In [ ]:
# Select features used for kNN
features = ['startYear', 'genres', 'averageRating', 'numVotes']
X = df.loc[:, features].copy()

## Clean Data

In [ ]:
# Remove duplicates
X = X.drop_duplicates()

## Construct Data

### Log-Scaling of feature `numVotes`

In [ ]:
X["log_numVotes"] = np.log10(X["numVotes"])
X = X.drop(columns=["numVotes"])

### Multi-Hot Encoding of feature `genres`

In [ ]:
# Turn genres into multi-hot encoding
if 'genres' in features:
    X['genres'] = X['genres'].apply(str.split, args=(','))
    mlb = MultiLabelBinarizer()
    X = X.join(pd.DataFrame(mlb.fit_transform(X.pop('genres')), columns='genre'+mlb.classes_, index=X.index))

### ❗ **Homework** ❗  
2. Investigate how the training DataFrame (`X`) looks like compared to the original DataFrame (`df`). What do you notice?


In [ ]:
# ...

**Observations:**  
- ...

# Step 4: Modeling

## Create Pipeline

### ❗ **Homework** ❗  
3.
Create a pipeline for the kNN model. Use a standard scaler and the `NearestNeighbors` module.


In [ ]:
# ...

## Recommend Movies

### ❗ **Homework** ❗  
4. Generate search results for movies similar to 'The Dark Knight'. Use the 'Search' example from the kNN hands-on to fill out the blanks.


In [ ]:
movie_name = 'The Dark Knight'

# Create dataframe with new example (picked from the data set)
sample_df = X.loc[df.loc[df.originalTitle == movie_name].index]

In [ ]:
# Transform and find  5 nearest neighbors
transformed_data_point = ...
distances, indices = ...

# Show dataframe with movie suggestions
for idx in indices:
    display(df.loc[X.iloc[idx].index])